In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("fifa_world_cup_2026.csv")

In [3]:
df.head()

,player_id,player_name,age,nationality,team,jersey_number,position,height_cm,weight_kg,preferred_foot,...,possession_impact,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards,tournament_rating
0,P00055,Rodri Fati,26,Spanish,Spain,3,Goalkeeper,195,75,Left,...,1.1,44.2,55.9,42.0,51.8,0,0,242,0,5.8
1,P00070,Ansu Le Normand,19,Spanish,Spain,18,Midfielder,178,75,Right,...,3.5,38.2,43.7,31.1,52.7,0,3,342,0,5.5
2,P00066,Gavi Ramos,18,Spanish,Spain,14,Midfielder,177,72,Left,...,15.3,99.0,99.0,83.4,54.8,1,1,245,0,8.4
3,P00073,Pedro Cubarsi,20,Spanish,Spain,21,Forward,182,74,Right,...,1.2,19.8,42.3,40.9,78.5,5,3,422,0,6.7
4,P00059,Alvaro Oyarzabal,23,Spanish,Spain,7,Defender,191,81,Left,...,6.2,44.1,33.5,60.0,56.6,0,0,440,0,5.7


In [4]:
df.drop(columns=[
    'player_id',
    'player_name',
    'jersey_number',
    'club_name',
    'market_value_eur',
    'match_id',
    'match_date',
    'stadium',
    'city',
    'total_goals_tournament',
    'total_assists_tournament',
    'total_minutes_tournament',
    'tournament_rating',
    'player_rating',
    'performance_score',
    'goals',
    'assists',
    'goals_team',
    'goals_opponent'
], inplace=True)

In [5]:
df.head()

,age,nationality,team,position,height_cm,weight_kg,preferred_foot,opponent_team,tournament_stage,match_result,...,decelerations,stamina_score,offensive_contribution,defensive_contribution,possession_impact,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,player_of_match_awards
0,26,Spanish,Spain,Goalkeeper,195,75,Left,South Africa,Group Stage,W,...,23,81.9,3.3,48.2,1.1,44.2,55.9,42.0,51.8,0
1,19,Spanish,Spain,Midfielder,178,75,Right,South Africa,Group Stage,W,...,17,85.5,37.9,29.4,3.5,38.2,43.7,31.1,52.7,0
2,18,Spanish,Spain,Midfielder,177,72,Left,South Africa,Group Stage,W,...,19,88.8,79.8,78.6,15.3,99.0,99.0,83.4,54.8,0
3,20,Spanish,Spain,Forward,182,74,Right,South Africa,Group Stage,W,...,19,89.2,47.3,6.9,1.2,19.8,42.3,40.9,78.5,0
4,23,Spanish,Spain,Defender,191,81,Left,South Africa,Group Stage,W,...,18,73.6,33.0,75.6,6.2,44.1,33.5,60.0,56.6,0


In [6]:
print(df['player_of_match_awards'].value_counts())

player_of_match_awards
0    53225
1     1124
2      209
3       36
4        6
Name: count, dtype: int64


In [7]:
df['player_of_match_awards'] = df['player_of_match_awards'].apply(
    lambda x: 0 if x == 0 else 1
)

In [8]:
X = df.drop('player_of_match_awards', axis=1)
y = df['player_of_match_awards']

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [23]:
from sklearn.preprocessing import LabelEncoder
import pickle

categorical_columns = [
    "nationality",
    "team",
    "position",
    "preferred_foot",
    "opponent_team",
    "tournament_stage",
    "match_result"
]

label_encoder = {}

for col in categorical_columns:
    le = LabelEncoder()

    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

    label_encoder[col] = le

In [24]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train, y_train = smote.fit_resample(X_train, y_train)

In [25]:
from sklearn.preprocessing import StandardScaler
import joblib

# Create scaler
scaler = StandardScaler()

# Fit on training data and transform
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)


In [26]:
from sklearn.svm import SVC

model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    random_state=42
)

In [27]:
model.fit(X_train_scaled, y_train)

SVC(class_weight='balanced', random_state=42)

In [28]:
y_pred = model.predict(X_test_scaled)

In [29]:
y_test

20754    0
17552    0
51059    0
16215    0
45436    0
        ..
25725    0
35016    0
23496    0
27088    0
4787     0
Name: player_of_match_awards, Length: 10920, dtype: int64

In [30]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [31]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

Accuracy : 0.9396520146520146
Precision: 0.08260869565217391
Recall   : 0.13818181818181818
F1 Score : 0.10340136054421768


In [32]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.96      0.97     10645
           1       0.08      0.14      0.10       275

    accuracy                           0.94     10920
   macro avg       0.53      0.55      0.54     10920
weighted avg       0.95      0.94      0.95     10920

[[10223   422]
 [  237    38]]


In [33]:
import joblib

joblib.dump(model, "svc_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("✅ All files saved successfully!")

✅ All files saved successfully!
